[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S03/S03_01_nosql_mongomock.ipynb)

# S03_01 · NoSQL con MongoDB (sin instalar nada): pymongo + mongomock
**Sprint 3 · Del dato al modelo en la nube · Laboratorio OPCIONAL de NoSQL (Python) · ⏱ 60 min**

## Qué vas a aprender
1. Pasar de tablas relacionadas a **documentos** (JSON) que llevan dentro lo que necesitan.
2. Insertar, consultar, actualizar y borrar documentos (CRUD) con `pymongo`.
3. Hacer **agregaciones** (el `GROUP BY` de MongoDB) y compararlas con SQL.
4. Decidir cuándo conviene SQL y cuándo NoSQL.

## Contexto TurisData
*TurisData Canarias* guarda hoy sus reservas en SQLite (`turisdata.db`). Un equipo de producto quiere una
ficha de reserva que se lea de un solo golpe (con datos del alojamiento y del cliente dentro) para una app
móvil. Vamos a modelar **el mismo dato** en documentos.

> **Sobre MongoDB de verdad.** Aquí usamos `mongomock`, una imitación en memoria de MongoDB: no hay que
> instalar ni crear cuentas, y el código de `pymongo` es el mismo. Al final verás cómo apuntarlo a un
> servidor real (Compass local o Atlas). Los datos son sintéticos.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
# Librerías (en Colab puede faltar mongomock: esta celda lo instala si hace falta; necesita internet)
import importlib.util, subprocess, sys
for paquete in ("pymongo", "mongomock"):
    if importlib.util.find_spec(paquete) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import sqlite3
import pandas as pd
import mongomock

SEMILLA = 42

## 1 · Idea: de filas relacionadas a un documento
En SQL, para ver una reserva con su alojamiento y su cliente hacemos un `JOIN` entre tres tablas.
En un modelo **documental** guardamos la reserva con esa información **embebida**:

```json
{"_id": 5760, "canal": "empresa", "noches": 3, "precio_noche": 89.24, "cancelada": 1,
 "alojamiento": {"id": 5, "nombre": "Alojamiento 05", "isla": "Tenerife", "categoria": "hotel"},
 "cliente": {"id": 1234, "pais": "UK"}}
```
Ventaja: una lectura, sin `JOIN`. Coste: el dato del alojamiento **se repite** en cada reserva (si cambia,
hay que actualizarlo en muchos sitios).

Construimos los documentos a partir de `turisdata.db` (una `JOIN` de SQL, una sola vez):

In [ ]:
con = sqlite3.connect(dato("turisdata.db"))
filas = pd.read_sql_query("""
    SELECT r.*, a.nombre AS aloj_nombre, a.isla AS aloj_isla, a.categoria AS aloj_categoria,
           a.plazas AS aloj_plazas, c.pais AS cli_pais
    FROM reservas r
    JOIN alojamientos a ON a.id_alojamiento = r.id_alojamiento
    JOIN clientes c ON c.id_cliente = r.id_cliente
""", con)

documentos = []
for f in filas.to_dict("records"):
    documentos.append({
        "_id": f["id_reserva"],                     # en Mongo la clave primaria se llama _id
        "fecha_reserva": f["fecha_reserva"], "fecha_llegada": f["fecha_llegada"],
        "noches": f["noches"], "canal": f["canal"], "tarifa": f["tarifa"], "deposito": f["deposito"],
        "tipo_habitacion": f["tipo_habitacion"], "precio_noche": f["precio_noche"],
        "cancelada": bool(f["cancelada"]),
        "alojamiento": {"id": f["id_alojamiento"], "nombre": f["aloj_nombre"], "isla": f["aloj_isla"],
                        "categoria": f["aloj_categoria"], "plazas": f["aloj_plazas"]},
        "cliente": {"id": f["id_cliente"], "pais": f["cli_pais"]},
    })
print(len(documentos), "documentos")
print(documentos[0])

## 2 · Idea: insertar en una colección
Una **colección** es como una tabla, pero sin esquema fijo. Una **base de datos** agrupa colecciones.
Con `mongomock.MongoClient()` todo vive en memoria. Con un servidor real cambiaría solo la línea del cliente.

In [ ]:
USAR_MONGO_REAL = False          # pon True y rellena MONGO_URI solo si tienes Compass local o Atlas
MONGO_URI = "mongodb://localhost:27017"   # [verificar] cadena de conexión de tu servidor o de Atlas

if USAR_MONGO_REAL:
    from pymongo import MongoClient
    cliente = MongoClient(MONGO_URI)
else:
    cliente = mongomock.MongoClient()

bd = cliente["turisdata"]
reservas = bd["reservas"]
reservas.delete_many({})                  # empezar limpio si repites la celda
reservas.insert_many(documentos)
print("Documentos en la colección:", reservas.count_documents({}))

### Ejercicio 1 · Contar documentos
Guarda en la variable `n_canceladas` cuántas reservas tienen `cancelada` igual a `True`.
Pista: `count_documents(filtro)` recibe un diccionario, por ejemplo `{"canal": "directo"}`.

In [ ]:
# TODO: count_documents con el filtro {"cancelada": True}
...
assert n_canceladas == 2754, "Revisa el filtro: debe contar las reservas con cancelada = True (deberían ser 2754)"
print("Correcto:", n_canceladas, "reservas canceladas")

## 3 · Idea: consultar con `find`
`find(filtro, proyección)` devuelve un cursor. Los filtros son diccionarios:

| En SQL | En MongoDB |
|---|---|
| `WHERE canal = 'directo'` | `{"canal": "directo"}` |
| `WHERE noches > 5` | `{"noches": {"$gt": 5}}` |
| `WHERE canal IN ('directo','agencia')` | `{"canal": {"$in": ["directo", "agencia"]}}` |
| `A AND B` | `{"A": ..., "B": ...}` (coma) o `{"$and": [...]}` |
| `A OR B` | `{"$or": [{...}, {...}]}` |
| campo de un subdocumento | `{"alojamiento.isla": "Tenerife"}` (con punto) |

In [ ]:
# Ejemplo resuelto: reservas de más de 10 noches en Lanzarote, solo algunos campos, las 3 más largas
cursor = (reservas
          .find({"alojamiento.isla": "Lanzarote", "noches": {"$gt": 10}},
                {"noches": 1, "canal": 1, "alojamiento.nombre": 1})   # 1 = incluir ese campo
          .sort("noches", -1)                                          # -1 = descendente
          .limit(3))
for doc in cursor:
    print(doc)

### Ejercicio 2 · Filtrar
Guarda en `caras_tenerife` la lista (usa `list(...)`) de reservas de alojamientos de **Tenerife** con
`precio_noche` mayor que 150. Después comprueba lo que has obtenido.

In [ ]:
# TODO: find con {"alojamiento.isla": ..., "precio_noche": {"$gt": ...}}
...
assert isinstance(caras_tenerife, list), "Envuelve el cursor con list(...)"
assert len(caras_tenerife) == 194, f"Revisa el filtro: se esperaban 194 reservas y hay {len(caras_tenerife)}"
assert all(d["alojamiento"]["isla"] == "Tenerife" and d["precio_noche"] > 150 for d in caras_tenerife)
print("Correcto:", len(caras_tenerife), "reservas")

### Ejercicio 3 · Operadores lógicos
Guarda en `n_riesgo` cuántas reservas son de canal `web_intermediaria` **o** de tarifa `flexible`, y además
tienen `deposito` igual a `"sin_deposito"`. Pista: `$or` dentro de un filtro que también pide el depósito.

In [ ]:
# TODO: {"$or": [{...}, {...}], "deposito": ...}
...
assert n_riesgo == 2682, f"Revisa el $or y el depósito: se esperaban 2682 y tienes {n_riesgo}"
print("Correcto:", n_riesgo)

## 4 · Idea: actualizar y borrar
- `update_one(filtro, {"$set": {...}})` cambia campos de **un** documento; `update_many` de todos los que cumplan el filtro.
- `$inc` suma a un número; `$unset` elimina un campo.
- `delete_one` / `delete_many` borran. **Como en SQL: sin filtro se afecta a todo.**
- Buena práctica: comprobar antes con `count_documents(mismo_filtro)`.

In [ ]:
# Ejemplo resuelto: añadir un campo nuevo a las reservas del canal empresa (en SQL habría que hacer ALTER TABLE)
antes = reservas.count_documents({"canal": "empresa"})
resultado = reservas.update_many({"canal": "empresa"}, {"$set": {"facturable": True}})
print(f"Filtradas: {antes} · modificadas: {resultado.modified_count}")
print(reservas.find_one({"canal": "empresa"}, {"canal": 1, "facturable": 1}))

### Ejercicio 4 · Actualizar
Sube 5 euros el `precio_noche` de la reserva con `_id` igual a 5760 usando `$inc`. Guarda el nuevo precio
leído de la base en `precio_nuevo`. (El original era 89.24.)

In [ ]:
# TODO: update_one({"_id": 5760}, {"$inc": {"precio_noche": 5}}) y luego find_one
...
assert round(precio_nuevo, 2) == 94.24, f"El precio debería ser 94.24 y es {precio_nuevo}. ¿Ejecutaste el update una sola vez?"
print("Correcto:", precio_nuevo)

### Ejercicio 5 · Borrar con cuidado
Primero guarda en `a_borrar` cuántas reservas hay con `noches` igual a 1 y `cancelada` `True`; después
bórralas con `delete_many` y guarda en `quedan` cuántas quedan con ese mismo filtro (debe ser 0).

In [ ]:
# TODO: usa el mismo filtro en count_documents y en delete_many
...
assert a_borrar > 0, "Antes de borrar tenía que haber alguna reserva que cumpliera el filtro"
assert quedan == 0, "Tras el delete_many no debería quedar ninguna"
assert reservas.count_documents({}) == 6000 - a_borrar, "Se han borrado más documentos de los previstos"
print(f"Correcto: borradas {a_borrar} reservas")

## 5 · Idea: agregaciones (el `GROUP BY` de MongoDB)
Una **pipeline** es una lista de etapas; cada una transforma el resultado de la anterior:

| Etapa | Equivale a |
|---|---|
| `$match` | `WHERE` |
| `$group` | `GROUP BY` con `$sum`, `$avg`, `$min`, `$max` |
| `$sort`, `$limit` | `ORDER BY`, `LIMIT` |
| `$project` | elegir/calcular columnas |
| `$unwind` | "desenrollar" un array en varios documentos |

Nota: tras el ejercicio 5 hemos borrado reservas, así que para comparar con SQL recargamos la colección.

In [ ]:
reservas.delete_many({})
reservas.insert_many(documentos)

# Ejemplo resuelto: número de reservas y precio medio por tipo de habitación
pipeline = [
    {"$group": {"_id": "$tipo_habitacion", "reservas": {"$sum": 1}, "precio_medio": {"$avg": "$precio_noche"}}},
    {"$sort": {"precio_medio": -1}},
]
for fila in reservas.aggregate(pipeline):
    print(fila["_id"], fila["reservas"], round(fila["precio_medio"], 2))

### Ejercicio 6 · Tasa de cancelación por canal
Crea la lista `tasa_canal` con la pipeline que agrupa por `canal` y calcula el número de reservas y el
número de canceladas (`$sum` de una condición: `{"$cond": ["$cancelada", 1, 0]}`). Después calculamos el % con pandas.

In [ ]:
# TODO: $group por "$canal" con dos $sum: {"$sum": 1} y {"$sum": {"$cond": ["$cancelada", 1, 0]}}
...
assert len(tasa_canal) == 4, "Debe haber una fila por canal (4 canales)"
tabla = pd.DataFrame(tasa_canal).rename(columns={"_id": "canal"}).set_index("canal")
tabla["pct_cancelacion"] = (100 * tabla["canceladas"] / tabla["reservas"]).round(1)
print(tabla.sort_values("pct_cancelacion", ascending=False))
assert tabla["reservas"].sum() == 6000, "La suma de reservas debe ser 6000"

### Ejercicio 7 · Ingreso por isla (con subdocumentos)
Calcula en `ingreso_isla` (lista) el ingreso estimado (`noches * precio_noche`) de las reservas **no
canceladas**, agrupado por `alojamiento.isla`. Etapas: `$match` → `$group` con `{"$multiply": ["$noches", "$precio_noche"]}`.

In [ ]:
# TODO: $match {"cancelada": False}; $group por "$alojamiento.isla" con $sum de $multiply
...
assert len(ingreso_isla) == 6, "Debe haber una fila por isla (6 islas)"
print(pd.DataFrame(ingreso_isla).rename(columns={"_id": "isla"}).round(0))

## 6 · Idea: la flexibilidad del documento (arrays y campos opcionales)
Un documento puede llevar **listas**: por ejemplo las experiencias contratadas con la reserva. En SQL
necesitaríamos una tabla intermedia (ver `06_modelado_indices.sql`). Aquí basta un array, y `$unwind`
lo "desenrolla" para contarlo.

In [ ]:
reservas.update_one({"_id": 5760}, {"$set": {"experiencias": [
    {"nombre": "Excursión al Teide", "precio": 45.0}, {"nombre": "Avistamiento de cetáceos", "precio": 38.0}]}})
reservas.update_one({"_id": 3674}, {"$set": {"experiencias": [{"nombre": "Excursión al Teide", "precio": 45.0}]}})

for fila in reservas.aggregate([
    {"$unwind": "$experiencias"},
    {"$group": {"_id": "$experiencias.nombre", "contratos": {"$sum": 1}, "importe": {"$sum": "$experiencias.precio"}}},
    {"$sort": {"contratos": -1}},
]):
    print(fila)

## 7 · Mini-reto integrador: la misma pregunta en SQL y en MongoDB
Pregunta del cliente: **¿cuál es la tasa de cancelación (%) de cada isla?**
1. Escribe la consulta SQL sobre `turisdata.db` y guarda el resultado en `tasa_sql` (diccionario `{isla: %}` con 1 decimal).
2. Escribe la pipeline de MongoDB y guarda el resultado en `tasa_mongo` (mismo formato).
3. Comprueba que coinciden.

In [ ]:
# TODO: SQL: JOIN reservas-alojamientos, GROUP BY isla, AVG(cancelada); Mongo: $group por "$alojamiento.isla"
...
assert set(tasa_sql) == set(tasa_mongo) and len(tasa_sql) == 6, "Deben salir las 6 islas en ambos casos"
assert tasa_sql == tasa_mongo, "Los resultados de SQL y de MongoDB deberían coincidir; compara isla a isla"
print("¡Coinciden! ", tasa_sql)

## 8 · Para reflexionar (respóndelo en tu cuaderno de aprendizaje)
1. ¿Qué ventaja tiene guardar el alojamiento **dentro** de cada reserva? ¿Y qué problema si un alojamiento cambia de nombre?
2. ¿Para qué tipo de datos de TurisData usarías MongoDB (reseñas, registros de navegación, catálogo de experiencias con atributos distintos) y para cuáles seguirías con SQL (facturas)?
3. Con `mongomock` no hay servidor: ¿qué cosas tendrías que pensar además al pasar a un MongoDB real (usuarios, copias, índices, coste)?

## 9 · Qué has aprendido
- Un **documento** agrupa lo que se lee junto; una **colección** agrupa documentos parecidos.
- CRUD: `insert_many`, `find`, `update_one/many` con `$set`/`$inc`, `delete_many`.
- Agregaciones: `$match`, `$group`, `$sort`, `$unwind`; equivalen a `WHERE`, `GROUP BY`, `ORDER BY`.
- SQL y MongoDB dan **la misma respuesta**; cambia el modelo de datos y las ventajas de cada uno.

**Para pasar a MongoDB de verdad:** pon `USAR_MONGO_REAL = True`, sustituye `MONGO_URI` por tu cadena de conexión y ejecuta de nuevo (consulta `guias/S03/NoSQL_MongoDB.md`).